# Les 12 - Vermindering van Chatgeschiedenis met Agent Scratchpad

Deze notebook toont hoe je context beheert in lange gesprekken met behulp van Microsoft Agent Framework. Naarmate gesprekken groeien, neemt het aantal tokens toe — en overschrijdt uiteindelijk het contextvenster van het model. We lossen dit op met een **context-samenvattingspatroon** en een **agent scratchpad** voor persistente geheugen.

## Wat je zult leren:
1. **Waarom Contextbeheer Belangrijk Is**: Begrijpen van tokenlimieten en contextvensters
2. **Contextbewuste Agents**: Agents bouwen die hun eigen gesprekscontext beheren
3. **Context-samenvattingspatroon**: Tools gebruiken om de gesprekshistorie samen te vatten
4. **Agent Scratchpad**: Persistente geheugen dat contextvermindering overleeft

## Vereisten:
- Azure OpenAI ingesteld met geconfigureerde omgevingsvariabelen
- Begrip van basisagentconcepten uit eerdere lessen


## Installatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Waarom Contextbeheer Belangrijk Is

Elke LLM heeft een eindig **contextvenster** — het maximale aantal tokens dat het in één verzoek kan verwerken. Naarmate een meer beurten gesprek vordert:

- **Het aantal tokens groeit lineair** met elk gebruikersbericht en assistentantwoord.
- **Prompttokens domineren de kosten** omdat de gehele geschiedenis bij elke beurt opnieuw wordt verzonden.
- Uiteindelijk **overschrijdt het gesprek het contextvenster** en het model verkort dan of geeft een foutmelding.

### Strategieën voor Contextbeheer

| Strategie | Hoe het Werkt | Afweging |
|---|---|---|
| **Verkorting** | Oudste berichten verwijderen | Verliest vroege context |
| **Samenvatting** | Oudere berichten samenvatten | Sommige details gaan verloren, maar belangrijkste punten blijven behouden |
| **Kladblok / Extern Geheugen** | Belangrijke feiten buiten het gesprek opslaan | Vereist tool-aanroepen, maar overleeft elke reductie |

In dit notitieboek combineren we **samenvatting** met een **kladbloktool** zodat de agent continuïteit kan behouden, zelfs wanneer de gesprekshistorie wordt ingekort.


## Een contextbewuste agent creëren


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Een Lang Gesprek Simuleren

Laten we een gesprek met meerdere beurten doorlopen om te zien hoe context zich ophoopt. De agent moet belangrijke details (voorkeuren, budget, reisdata) over beurten heen onthouden en continuïteit aantonen.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Let op hoe de agent context behoudt van eerdere beurten — hij weet van Japan, sushi, tempels, fotografie, het budget van $3000, solo reizen en de periode in april. In een kort gesprek werkt dit goed, maar naarmate het gesprek groeit, wordt de volledige geschiedenis kostbaar om telkens opnieuw te versturen.

Laten we het gesprek voortzetten met meer beurten om contextopbouw te zien:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Samenvatting Patroon

Naarmate het gesprek groeit, kunnen we een **samenvattingshulpmiddel** gebruiken om de opgebouwde context in een compact formaat samen te vatten. De agent roept dit hulpmiddel aan om belangrijke voorkeuren vast te leggen, zodat zelfs als oudere berichten worden verwijderd, de essentiële informatie behouden blijft.

Dit patroon is de bouwsteen voor meer geavanceerde geschiedenisreductie:
1. De agent identificeert belangrijke feiten uit het gesprek
2. Hij roept het samenvattingshulpmiddel aan om deze te bewaren
3. Oudere berichten kunnen veilig worden verwijderd omdat de samenvatting vastlegt wat belangrijk is

Hieronder definiëren we een `summarize_preferences` hulpmiddel dat de agent kan aanroepen om een compacte samenvatting op te slaan van wat hij heeft geleerd.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Samenvatting

In deze les heb je geleerd hoe je context beheert in langdurige agentgesprekken met behulp van het Microsoft Agent Framework:

### Belangrijke Concepten
- **Contextvensters zijn beperkt** — elke token in de gespreksgeschiedenis kost geld en telt mee voor de limiet.
- **Samenvattingstools** laten de agent toe om verzamelde context in compacte samenvattingen te condenseren, waardoor tokengebruik vermindert terwijl essentiële informatie behouden blijft.
- **Agent-krabbels** bieden persistent extern geheugen dat elke conversiereductie overleeft.

### Wat Je Hebt Gemaakt
- Een **contextbewuste agent** die continuïteit behoudt over gesprekken met meerdere beurten
- Een **samenvattingstool** (`summarize_preferences`) die belangrijke gebruikersgegevens op een compacte wijze vastlegt
- Een **gesprek met meerdere beurten** die contextbehoud en veranderingsafhandeling demonstreert

### Toepassingen in de Praktijk
- **Klantenservice-bots**: onthoud voorkeuren gedurende lange ondersteuningssessies
- **Persoonlijke assistenten**: volg lopende projecten zonder de context opnieuw uit te leggen
- **Educatieve tutors**: behoud voortgang van studenten over vele interacties

### Volgende Stappen
- Implementeer een volledige krabbel-tool met bestand-gebaseerde persistentie
- Voeg automatische historiekverkorting toe na samenvatten
- Combineer met vectordatabases voor semantisch geheugen zoeken
- Bouw agents die gesprekken na dagen kunnen hervatten met volledige context


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI-vertalingsdienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het oorspronkelijke document in de oorspronkelijke taal dient als de gezaghebbende bron te worden beschouwd. Voor belangrijke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
